In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

data = {
    'name':       ['Alice','Bob','Carol','Dave','Eve','Frank','Grace','Hank','Iris','Jack',
                   'Karen','Leo','Mia','Ned','Olivia','Pete','Quinn','Rose','Sam','Tina'],
    'dept':       ['Sales','Tech','Sales','HR','Tech','Sales','Tech','HR','Sales','Tech',
                   'HR','Sales','Tech','HR','Sales','Tech','Sales','HR','Tech','Sales'],
    'age':        [25, 32, 28, 45, 36, 52, 29, 41, 33, 38, 27, 60, 31, 44, 26, 35, 48, 39, 30, 55],
    'experience': [ 2,  8,  4, 20, 12, 28,  5, 18,  9, 14,  3, 35,  7, 21,  1, 11, 24, 16,  6, 30],
    'salary':     [42, 85, 47, 61, 90, 52, 88, 63, 45, 91, 58, 54, 86, 65, 44, 89, 50, 67, 82, 200],  # 200 = outlier
    'rating':     [ 4,  5,  3,  4,  5,  3,  5,  4,  4,  5,  3,  4,  5,  3,  4,  5,  4,  3,  5,  4],
    'sales_calls':[ 40,  0, 38,  0,  0, 45,  0,  0, 42,  0,  0, 35,  0,  0, 39,  0, 44,  0,  0, 41],
    'deals_won':  [ 12,  0, 10,  0,  0, 15,  0,  0, 11,  0,  0,  9,  0,  0, 13,  0, 14,  0,  0, 12],
    'join_date':  ['2022-03-15','2016-07-01','2020-11-20','2003-05-10','2012-01-25',
                   '1996-08-14','2019-04-03','2006-02-28','2015-09-17','2010-06-12',
                   '2021-12-01','1989-03-22','2017-08-30','2003-11-05','2023-01-10',
                   '2013-07-19','2000-04-27','2008-10-03','2018-02-14','1994-06-30'],
}

df = pd.DataFrame(data)
df['join_date'] = pd.to_datetime(df['join_date'])
# salary in thousands; age/experience in years; rating 1-5; sales_calls & deals_won for Sales dept only
df.head()

# Segmented Univariate Analysis

Compare the distribution of a **numeric variable** across different **subgroups** (categories).

Same as univariate, but done separately for each segment — reveals patterns hidden in overall summaries.

---

## The Core Tool — groupby + describe

In [ ]:
df.groupby('category_col')['numeric_col'].describe()

Returns count, mean, std, min, Q1, Q2, Q3, max **for each category**.

### Examples from NAS education dataset

In [ ]:
# How does TV watching relate to science scores?
df.groupby('Watch.TV')['Science..'].describe()

# Does father's education affect maths scores?
df.groupby('Father.edu')['Maths..'].describe()

# Mother's education vs reading scores
df.groupby('Mother.edu')['Reading..'].describe()
# Result: Degree & above → mean 70 vs Illiterate → mean 49

### Other aggregations

In [ ]:
df.groupby('cat')['num'].mean()     # just the mean per group
df.groupby('cat')['num'].median()
df.groupby('cat')['num'].agg(['mean', 'std', 'count'])

---

## Visualisation — Grouped Plots

### Box plot by category (best for segmented analysis)

In [ ]:
sns.boxplot(data=df, x='category_col', y='numeric_col')

Shows median, IQR, outliers for each group side by side — easy to compare spread and centre.

### Histogram per group

In [ ]:
# overlapping histograms
sns.histplot(data=df, x='numeric_col', hue='category_col', bins=10)

### Violin plot — box plot + density shape

In [ ]:
sns.violinplot(data=df, x='category_col', y='numeric_col')

Wider sections = more data there; shows full distribution shape per group.

---

## What to Look For

| Pattern | Interpretation |
|---------|---------------|
| Means differ significantly across groups | The category has an effect on the numeric variable |
| Medians differ but means don't | Skewed distributions per group — look at median |
| One group has much wider IQR | More variability in that segment |
| Mean increases monotonically across ordered categories | Likely a real relationship (e.g. higher education → higher scores) |

---

## Subsetting for Comparison

In [ ]:
# Filter to a specific segment then analyse
segment = df[df['category_col'] == 'value']
segment['numeric_col'].describe()

---

## Example — Interview Candidate Tiering with Q1/Q3

In [ ]:
# Get Q1 and Q3 for all numeric subjects at once
q1 = df.select_dtypes('number').quantile(0.25)
q3 = df.select_dtypes('number').quantile(0.75)

# Direct to final: above Q3 in ALL subjects
final  = df[(df[subjects] >= q3).all(axis=1)]

# Second stage: between Q1 and Q3 in ALL subjects
second = df[(df[subjects] >= q1).all(axis=1) & (df[subjects] < q3).all(axis=1)]

# Reject: below Q1 in ANY subject
reject = df[(df[subjects] < q1).any(axis=1)]